# Week 6, Day 3: Baseline and scikit-learn pricers for memorabilia

**Lesson:** [week6/day3.ipynb](../../../week6/day3.ipynb), *Evaluation, baselines and traditional ML*. **About 25 minutes.**

**What you practice here**

- A baseline that every model must beat, written as a pricer: the lesson's constant pricer, which always guesses the average training price.
- Turning an item into numeric features, with a flag for a missing value, as the lesson's `get_features` flags an unknown weight.
- scikit-learn's `fit` and `predict` on a table of features, wrapped as a pricer that predicts from a one-row table.
- A bag of words: `fit_transform` on the training texts, `transform` on a new one, any regressor behind it, and the price kept at 0 or above.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week6/day3", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson prices products with simple models before it tries any LLM: a constant baseline, a linear regression on a
few hand-made features, and models on a bag of words. Each becomes a pricer, a function from an item to a price,
which the lesson's `evaluate` scores. Build the same pricers for wrestling memorabilia listings. Each listing has a
`title`, a `summary` in Day 2's five-line format, a `price`, a `year` (`None` when it is unknown) and `signed`
(`True` or `False`).

scikit-learn is not in the checks, so the two functions that train a model take it as a parameter, and
`fit_text_pricer` takes its vectorizer too: you pass in objects such as `LinearRegression()` and
`CountVectorizer(max_features=2000, stop_words="english")`, and your code calls their methods. The checks pass
stand-ins with the same methods, which record what you send them.

## What each call should give

```python
pricer = make_constant_pricer(train)      # three listings, priced 85.00, 399.99 and 600.00
pricer(poster)
# 361.66333333333336, whatever the listing

features(program)                         # from 1987, not signed, with a summary of 212 characters
# {'year': 1987, 'year_unknown': 0, 'signed': 0, 'text_length': 212}

pricer = fit_feature_pricer(LinearRegression(), train, features)
pricer(poster)
# the model's prediction for the poster's one row of features, such as 104.2

pricer = fit_text_pricer(CountVectorizer(max_features=2000, stop_words="english"), LinearRegression(), train)
pricer(poster)
# the model's prediction for the poster's summary, never below 0, such as 57.9
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `make_constant_pricer(train)`

Builds the lesson's constant baseline: a pricer that always guesses the average training price. Work out the average
price of the `train` listings once, and return a function that takes a listing and returns that average. The pricer
must not look at the listing it prices, not even at its price.

<details><summary>Hint</summary>

The lesson works out `training_average` first, and its `constant_pricer` only returns it. Here the pricer is a
function defined inside `make_constant_pricer`, which then returns it:

```python
def constant_pricer(item):
    return training_average
```

</details>

In [ ]:
def make_constant_pricer(train):
    # Write make_constant_pricer here, as described above.
    raise NotImplementedError("make_constant_pricer is not written yet")

In [ ]:
check("make_constant_pricer")

### 2 · `features(listing)`

Turns a listing into numeric features, as the lesson's `get_features` does for its items. Return a dict with exactly
these keys, in this order: `"year"` (the year, or 0 when it is `None`), `"year_unknown"` (1 when the year is
`None`, otherwise 0), `"signed"` (1 when the listing is signed, otherwise 0) and `"text_length"` (the number of
characters in `listing.summary`).

<details><summary>Hint</summary>

The lesson flags its missing weight with `1 if item.weight == 0 else 0`. Here the missing value is `None`:
`listing.year or 0` gives the year or 0, and `1 if listing.year is None else 0` gives the flag.

</details>

In [ ]:
def features(listing):
    # Write features here, as described above.
    raise NotImplementedError("features is not written yet")

In [ ]:
check("features")

### 3 · `fit_feature_pricer(model, train, features)`

Trains a regression on a table of features and returns it as a pricer, as the lesson does with its linear
regression. `model` is a scikit-learn regressor such as `LinearRegression()`, and `features` is a function like
yours from part 2. Build the table, one row per training listing with the values of `features(listing)` in order,
and the list of their prices, then fit the model on them once. Return a pricer that turns one listing into a table of
one row, has the model predict on it, and returns the first (and only) number of the prediction. The pricer must not
fit again.

<details><summary>Hint</summary>

scikit-learn takes a table as a list of rows, and predicts one value per row:

```python
X = [list(features(item).values()) for item in train]
model.fit(X, [item.price for item in train])
# in the pricer:
return model.predict([list(features(item).values())])[0]
```

</details>

In [ ]:
def fit_feature_pricer(model, train, features):
    # Write fit_feature_pricer here, as described above.
    raise NotImplementedError("fit_feature_pricer is not written yet")

In [ ]:
check("fit_feature_pricer")

### 4 · `fit_text_pricer(vectorizer, model, train)`

Trains a model on a bag of words and returns it as a pricer, as the lesson does with its `CountVectorizer`.
`vectorizer` is a scikit-learn text vectorizer such as `CountVectorizer(max_features=2000, stop_words="english")`,
and `model` is any regressor: `LinearRegression`, `RandomForestRegressor` and `XGBRegressor` all work the same way.
Fit the vectorizer on the training summaries, in order, turning them into a table as it learns their words, then fit
the model on that table and the training prices. Return a pricer that turns one listing's summary into a row with the
fitted vectorizer (hand it a list holding the one text, and never fit it again, which would learn the new listing's
words), has the model predict on that row, and returns the first number, or 0 when it is below 0, as the lesson
does.

<details><summary>Hint</summary>

```python
X = vectorizer.fit_transform([item.summary for item in train])
model.fit(X, [item.price for item in train])
# in the pricer:
x = vectorizer.transform([item.summary])
return max(0, model.predict(x)[0])
```

</details>

In [ ]:
def fit_text_pricer(vectorizer, model, train):
    # Write fit_text_pricer here, as described above.
    raise NotImplementedError("fit_text_pricer is not written yet")

In [ ]:
check("fit_text_pricer")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 6, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Nothing here calls a model, so you need no key. It needs the course environment (`scikit-learn` and `datasets`), and fitting the random forest takes a few minutes.

1. Put the course's `week6` folder on the path with `sys.path.insert(0, str(ROOT / "week6"))`, load the lesson's
   data with `Item.from_hub("ed-donner/items_lite")`, and import the lesson's `evaluate`.
2. Score your baseline: `evaluate(make_constant_pricer(train), test)`.
3. Write a features function for the lesson's items (the weight, a flag for an unknown weight, and the text length),
   and score `fit_feature_pricer(LinearRegression(), train, item_features)`.
4. Score `fit_text_pricer` with `CountVectorizer(max_features=2000, stop_words="english")` and
   `LinearRegression()`, then with `RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)` on the first
   5,000 training items.

<details><summary>Stuck? One way to write it</summary>

```python
import sys

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression

sys.path.insert(0, str(ROOT / "week6"))  # the lesson's pricer package
from pricer.evaluator import evaluate
from pricer.items import Item

train, val, test = Item.from_hub("ed-donner/items_lite")
evaluate(make_constant_pricer(train), test)


def item_features(item):
    return {"weight": item.weight or 0, "weight_unknown": 1 if not item.weight else 0,
            "text_length": len(item.summary)}


evaluate(fit_feature_pricer(LinearRegression(), train, item_features), test)
evaluate(fit_text_pricer(CountVectorizer(max_features=2000, stop_words="english"), LinearRegression(), train), test)
forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
evaluate(fit_text_pricer(CountVectorizer(max_features=2000, stop_words="english"), forest, train[:5_000]), test)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.